# 03 · Clean, decide, reconcile

**Produces:** `data/clean/bookings_clean.csv.gz` (the table every later notebook uses), `data/clean/rejects_step5.csv.gz` (every row we removed, with the rule that removed it), `data/clean/decisions_log_auto.csv` (one row per rule, with its effect on the headline) and `data/clean/person_map.csv.gz` (patient id to person).

**Needs:** notebook 02 run.

**The rules of this notebook.**
1. Every cleaning step is a *decision* with a reason, a row count and an effect on the headline growth.
2. After every step: input = kept + rejected.
3. After every join: the row count is checked, and a join that grows is explained or undone.

## CONFIG
Everything that depends on the machine or the period lives in this one cell, so nothing else needs editing.

In [ ]:
# ---------- imports ----------
import gzip
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# show more columns when we print a dataframe
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

# ---------- paths ----------
# the notebooks live in the notebooks folder, so the repo root is one folder up
if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

RAW = ROOT / "data" / "raw"       # the data pack as we received it (we never edit this)
CLEAN = ROOT / "data" / "clean"   # the files our notebooks write
SLIDES = ROOT / "slides"          # charts and the excel file for the slides
CLEAN.mkdir(parents=True, exist_ok=True)
SLIDES.mkdir(parents=True, exist_ok=True)

# ---------- the two periods we compare ----------
H1_2025 = ("2025-01-01", "2025-06-30 23:59:59")   # 23:59:59 so that 30 June is included
H1_2026 = ("2026-01-01", "2026-06-30 23:59:59")

# a small helper: prints PASS or FAIL but never stops the notebook
def check(name, condition, detail=""):
    if condition:
        print("PASS", name, detail)
    else:
        print("FAIL", name, detail)

print("repo root:", ROOT)

In [ ]:
# ---------- load the stacked bookings from notebook 02 ----------
df = pd.read_csv(CLEAN / "bookings_all.csv.gz", parse_dates=["created_at", "slot_at"],
                 dtype={"booking_id": "str", "patient_id": "str", "test_code": "str",
                        "status_at_export": "str", "rescheduled_from": "str"}, low_memory=False)
print("shape:", df.shape)
print(df.dtypes)
df.head()

In [ ]:
# a small function for the headline: tests in each half-year and the growth between them
def headline(df_in):
    n_2025 = df_in["slot_at"].between(H1_2025[0], H1_2025[1]).sum()
    n_2026 = df_in["slot_at"].between(H1_2026[0], H1_2026[1]).sum()
    growth = (n_2026 / n_2025 - 1) * 100
    return n_2025, n_2026, growth

n_2025, n_2026, growth = headline(df)
print("BEFORE any cleaning: H1 2025 =", n_2025, " H1 2026 =", n_2026, " growth =", round(growth, 2), "%")

# the plan, wide: one column per month. we need its total
plan_wide = pd.read_csv(RAW / "plan" / "plan_2026_H1.csv")
plan_total = plan_wide.drop(columns=["city", "metric"]).sum().sum()
print("plan total for H1 2026:", plan_total)

## 1 · Collections: remove double scans, then join to the bookings
The phlebotomy app sometimes records the same scan twice. The two rows are exactly equal, so removing one is safe.

In [ ]:
coll_files = sorted((RAW / "collections").glob("*.csv.gz"))
coll_list = []
for path in coll_files:
    coll_list.append(pd.read_csv(path, dtype=str))
df_coll = pd.concat(coll_list, ignore_index=True)
print("collections files:", len(coll_files), " rows:", len(df_coll))

# strip spaces in the id, then remove rows that are exactly the same
df_coll["booking_id"] = df_coll["booking_id"].str.strip()
n_before = len(df_coll)
df_coll = df_coll.drop_duplicates()
print("double scans removed:", n_before - len(df_coll))

# after this every booking should have at most one collection row
check("one collection row per booking", df_coll["booking_id"].is_unique, str(df_coll["booking_id"].duplicated().sum()) + " repeated")
df_coll["booking_id"].duplicated().sum()

In [ ]:
# if some ids are still repeated (different status for the same booking) keep the first and say so
n_repeated = df_coll["booking_id"].duplicated().sum()
if n_repeated > 0:
    print("keeping the first row for", n_repeated, "booking ids that appear twice with different values")
    df_coll = df_coll.drop_duplicates(subset="booking_id", keep="first")

# join: every booking row keeps its place (LEFT join). validate makes pandas stop if the key is not unique
df_coll_small = df_coll[["booking_id", "status"]].rename(columns={"status": "coll_status"})
n_rows_before = len(df)
df = df.merge(df_coll_small, on="booking_id", how="left", validate="many_to_one")
check("LEFT join bookings to collections did not change the row count", len(df) == n_rows_before,
      str(n_rows_before) + " -> " + str(len(df)))

# a booking without a collection row was probably cancelled. we label it, we do not delete it
df["coll_status"] = df["coll_status"].fillna("no_collection_row")

# look at it per booking (not per test row)
print(df.drop_duplicates("booking_id")["coll_status"].value_counts())
print()
print("share of bookings:")
print((df.drop_duplicates("booking_id")["coll_status"].value_counts(normalize=True) * 100).round(1))

> **Interview angle.** Explain `how="left"`, `validate="many_to_one"` and why we check the row count after a join.
>
> *A good answer:* LEFT keeps every left row even without a match, like a SQL LEFT JOIN. `validate` makes pandas raise an error if the right side has repeated keys, which SQL would not tell you. The row count check is the cheapest way to see a join that multiplied rows (fan-out).

## 2 · Which rule for "a test"? (the count rule)
The plan was built from *some* count. We compute the growth three ways and see which one lands near 5% and near the plan's basis. The key column is `plan_total / H1_2025`: the plan was built as last year multiplied by a growth factor, so the rule that gives about 1.18 is the one the plan used.

In [ ]:
# first remove the old copies of rescheduled visits, so all three rules start from the same rows
is_old_copy = df["status_at_export"].eq("rescheduled").fillna(False).astype(bool)
df_start = df[~is_old_copy]

rule_tables = {
    "A  all booking-test rows": df_start,
    "B  collected only": df_start[df_start["coll_status"] == "collected"],
    "C  collected + no_show": df_start[df_start["coll_status"].isin(["collected", "no_show"])],
}

rows = []
for name in rule_tables:
    t = rule_tables[name]
    a, b, g = headline(t)
    rows.append({"rule": name, "H1_2025": a, "H1_2026": b, "growth_pct": round(g, 2), "plan_total / H1_2025": round(plan_total / a, 3)})
df_rules = pd.DataFrame(rows)
print(df_rules.to_string(index=False))
df_rules.to_csv(CLEAN / "count_rule_comparison.csv", index=False)

In [ ]:
# WRITE YOUR DECISION HERE after reading the table above (and put it in docs/approach-log.md)
#   "A" = count every booking-test row (including no-shows and cancellations)
#   "B" = count only samples that were collected
#   "C" = count collected and no-show, but not cancelled
COUNT_RULE = "A"
print("count rule chosen:", COUNT_RULE)

## 3 · Cleaning rules, one at a time
Each rule goes through the same function. It removes the rows, keeps them in a separate table, and records how the growth number moved.

In [ ]:
decisions = []        # one row per rule
rejected_list = []    # the rows each rule removed

def apply_rule(df_in, name, reason, reject_mask):
    n25_before, n26_before, growth_before = headline(df_in)

    df_rejected = df_in[reject_mask].copy()
    df_rejected["reject_rule"] = name
    df_kept = df_in[~reject_mask].copy()

    n25_after, n26_after, growth_after = headline(df_kept)
    check(name + ": input = kept + rejected", len(df_in) == len(df_kept) + len(df_rejected),
          str(len(df_in)) + " = " + str(len(df_kept)) + " + " + str(len(df_rejected)))

    decisions.append({"rule": name, "reason": reason, "rows_in": len(df_in), "rows_rejected": len(df_rejected),
                      "rows_out": len(df_kept), "H1_2025_after": n25_after, "H1_2026_after": n26_after,
                      "growth_before": round(growth_before, 2), "growth_after": round(growth_after, 2),
                      "moved_pts": round(growth_after - growth_before, 2)})
    rejected_list.append(df_rejected)
    return df_kept

In [ ]:
df_work = df.copy()

# ----- R1: the old copy of a rescheduled visit (nova only) -----
mask_r1 = df_work["status_at_export"].eq("rescheduled").fillna(False).astype(bool)
df_work = apply_rule(df_work, "R1 rescheduled originals (nova)",
                     "nova keeps the old booking and writes a new one; legacy overwrote the slot. one visit must be one booking",
                     mask_r1)

# ----- R2: nova bookings with no tests at all -----
mask_r2 = (df_work["system"] == "nova") & df_work["test_code"].isnull() & df_work["test_price"].isnull()
df_work = apply_rule(df_work, "R2 nova bookings with no tests",
                     "a booking whose test list is empty has no test to count", mask_r2)

# ----- R3: the count rule (only if we chose B or C) -----
if COUNT_RULE == "B":
    mask_r3 = df_work["coll_status"] != "collected"
    df_work = apply_rule(df_work, "R3 count rule B", "count performed tests only", mask_r3)
if COUNT_RULE == "C":
    mask_r3 = ~df_work["coll_status"].isin(["collected", "no_show"])
    df_work = apply_rule(df_work, "R3 count rule C", "count collected and no-show, not cancelled", mask_r3)

df_decisions = pd.DataFrame(decisions)
print(df_decisions.drop(columns="reason").to_string(index=False))

In [ ]:
# ----- things we only FLAG or REPORT, we do not remove them -----

# a blank test code with a valid price: the plan counted these rows, and we do not guess a code
df_work["flag_blank_test_code"] = df_work["test_code"].isnull() & df_work["test_price"].notnull()
print("rows with a blank test code but a price (kept, flagged):", df_work["flag_blank_test_code"].sum())

# rows that are exactly the same, in every column except the file name
dup_mask = df_work.drop(columns=["source_file"]).duplicated()
dup_2025 = (dup_mask & df_work["slot_at"].between(H1_2025[0], H1_2025[1])).sum()
dup_2026 = (dup_mask & df_work["slot_at"].between(H1_2026[0], H1_2026[1])).sum()
print("exact duplicate rows (reported, not removed): H1 2025 =", dup_2025, " H1 2026 =", dup_2026)

# the visit happened before the booking was made: probably a clock problem
clock_skew = df_work["slot_at"] < df_work["created_at"] - pd.Timedelta(days=1)
print("slot more than a day before the booking was created (flagged only):", clock_skew.sum())
df_work["flag_clock_skew"] = clock_skew

> **Do not fill missing values with guesses.** A blank test code could be matched to a price, but several codes share a price. Filling it would invent a test. We keep the row, flag it, count it, and say so in the log.
>
> **If the headline moved between two of our runs** (for example 436,302 to 435,430), the cause must be found and written in the decisions log. The exact-duplicate counts above are one thing to compare with the size of the difference.

## 4 · Patients: from ids to people
Merging two patient rows says "these two records are one human". For a growth analysis the cost of being wrong is a slightly wrong frequency figure. For a results system it would be a wrong report sent to the wrong person. Same rule, different stakes. We write the stakes in the log.

**Rule:** two rows are the same person if they have the same valid phone number and the same name (capitals and extra spaces ignored). A missing or fake phone is never merged. Name alone is never used, because common names belong to many different people.

In [ ]:
df_pat = pd.read_csv(RAW / "patients" / "patients.csv.gz", dtype=str)
df_pat["patient_id"] = df_pat["patient_id"].str.strip()
print(df_pat.shape)
print(df_pat.head(3))

# ---- phone: keep only the digits, then the last 10 ----
df_pat["phone_digits"] = df_pat["phone"].fillna("").str.replace(r"\D", "", regex=True)
df_pat["phone_norm"] = df_pat["phone_digits"].str[-10:]

# a phone is valid if it has 10 digits, is not one digit repeated (0000000000) and does not start with 000
ten_digits = df_pat["phone_norm"].str.len() == 10
all_same = df_pat["phone_norm"].str.fullmatch(r"(\d)\1{9}")
starts_000 = df_pat["phone_norm"].str.startswith("000")
df_pat["phone_ok"] = ten_digits & (~all_same) & (~starts_000)

# ---- name: capitals, one space between words ----
df_pat["name_norm"] = df_pat["patient_name"].fillna("").str.upper().str.replace(r"\s+", " ", regex=True).str.strip()

# ---- the key: phone + name when both are usable, otherwise the patient stays alone ----
can_merge = df_pat["phone_ok"] & (df_pat["name_norm"] != "")
df_pat["key"] = np.where(can_merge, df_pat["phone_norm"] + "|" + df_pat["name_norm"], "SINGLE|" + df_pat["patient_id"])
print("rows with a usable phone and name:", round(can_merge.mean() * 100, 1), "%")

In [ ]:
# the person id is the smallest patient id in the group
df_pat["person_id"] = df_pat.groupby("key")["patient_id"].transform("min")

n_ids = df_pat["patient_id"].nunique()
n_people = df_pat["person_id"].nunique()
print("patient ids:", n_ids, " people:", n_people, " ids merged:", n_ids - n_people, "(", round((n_ids - n_people) / n_ids * 100, 1), "% )")

# risk check: inside a merged group, do the dates of birth agree?
if "dob" in df_pat.columns:
    dob_ok = df_pat["dob"].where(~df_pat["dob"].fillna("").str.startswith("1900"))     # 1900 is a placeholder
    n_dob = df_pat[can_merge].assign(dob_ok=dob_ok).groupby("key")["dob_ok"].nunique()
    print("merged groups with more than one date of birth:", (n_dob > 1).sum(), "(possible different people)")

# the most common name: shows why we never merge by name alone
top_name = df_pat[can_merge]["name_norm"].value_counts().head(1)
print("most common name:", top_name.index[0], "on", top_name.iloc[0], "different patient ids")

In [ ]:
# do the merged ids ever appear in the bookings at all? (if not, merging cannot change the people count)
merged_ids = df_pat[df_pat["patient_id"] != df_pat["person_id"]]["patient_id"]
n_merged_in_bookings = merged_ids.isin(df_work["patient_id"]).sum()
print("merged ids:", len(merged_ids), " of them found in the bookings:", n_merged_in_bookings)

# attach person_id to every booking row
person_map = df_pat.drop_duplicates("patient_id")[["patient_id", "person_id"]]
n_before = len(df_work)
df_work["patient_id"] = df_work["patient_id"].str.strip()
df_work = df_work.merge(person_map, on="patient_id", how="left", validate="many_to_one")
check("joining person_id did not change the row count", len(df_work) == n_before)

# a patient id that is not in the master stays as its own person
n_missing = df_work["person_id"].isnull().sum()
print("booking rows whose patient id is not in the patient master:", n_missing)
df_work["person_id"] = df_work["person_id"].fillna(df_work["patient_id"])

# the effect on the number of people in each half
for name, period in [("H1 2025", H1_2025), ("H1 2026", H1_2026)]:
    d = df_work[df_work["slot_at"].between(period[0], period[1])]
    print(name, "patients counted as ids:", d["patient_id"].nunique(), " as people:", d["person_id"].nunique())

> **Interview angle.** What is the cost of being wrong when we merge two patient records, and who bears it?
>
> *A good answer:* in this project a wrong merge slightly moves visits-per-person, and we bounded it with a date-of-birth check. In a results system a wrong merge sends one patient's report to another person. The same rule needs a much stricter test when the stakes are higher.

## 5 · SUM: save the clean table and the logs

In [ ]:
df_rejected_all = pd.concat(rejected_list, ignore_index=True)
n_input = len(df)
n_clean = len(df_work)
n_rejected = len(df_rejected_all)
check("RECONCILE: input = clean + rejected", n_input == n_clean + n_rejected, str(n_input) + " = " + str(n_clean) + " + " + str(n_rejected))

n_2025, n_2026, growth = headline(df_work)
print("HEADLINE after cleaning: H1 2025 =", n_2025, " H1 2026 =", n_2026, " growth =", round(growth, 2), "%")

df_work.to_csv(CLEAN / "bookings_clean.csv.gz", index=False, compression="gzip")
df_rejected_all.to_csv(CLEAN / "rejects_step5.csv.gz", index=False, compression="gzip")
df_decisions.to_csv(CLEAN / "decisions_log_auto.csv", index=False)
person_map.to_csv(CLEAN / "person_map.csv.gz", index=False, compression="gzip")
print("saved bookings_clean.csv.gz, rejects_step5.csv.gz, decisions_log_auto.csv, person_map.csv.gz")

## Before you move on
- Copy every row of the decisions table above into `docs/decisions-log.md`, with the date, the owner, and the reason in our own words.
- Which rule moved the headline most? That rule goes on slide 5.
- Which rule would the finance controller challenge first?
- If the rows removed *before* our notebooks started (for example in an earlier merge) are not in this log, add them by hand: how many rows, and the growth before and after.